In [1]:
# Celda 1: Instalación (por si acaso falta algo)
!pip install torch torchaudio transformers datasets accelerate evaluate jiwer scikit-learn

     ---------------------------------------- 0.0/44.0 kB ? eta -:--:--
     --------- ------------------------------ 10.2/44.0 kB ? eta -:--:--
     -------------------------- ----------- 30.7/44.0 kB 435.7 kB/s eta 0:00:01
     ----------------------------------- -- 41.0/44.0 kB 487.6 kB/s eta 0:00:01
     -------------------------------------- 44.0/44.0 kB 359.1 kB/s eta 0:00:00
  Using cached datasets-4.4.2-py3-none-any.whl.metadata (19 kB)
     ---------------------------------------- 0.0/41.5 kB ? eta -:--:--
     ---------------------------------------- 41.5/41.5 kB 2.1 MB/s eta 0:00:00
  Using cached multiprocess-0.70.18-py311-none-any.whl.metadata (7.5 kB)
  Using cached click-8.3.1-py3-none-any.whl.metadata (2.6 kB)
  Using cached aiohttp-3.13.2-cp311-cp311-win_amd64.whl.metadata (8.4 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
   ---------------------------------------- 0.0/111.0 MB ? eta -:--:--
   ---------------------------------------- 0.1/111.0

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To up

In [2]:
# Celda 2: Importar librerías y configurar GPU
import os
import torch
import numpy as np
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Union
from datasets import load_from_disk
from transformers import (
    Wav2Vec2ForCTC,
    Wav2Vec2Processor,
    Trainer,
    TrainingArguments,
    set_seed
)
import evaluate

# Configurar semilla
set_seed(42)

# Verificar si tienes GPU
print(f"Torchaudio version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"🔥 GPU Detectada: {torch.cuda.get_device_name(0)}")
    device = "cuda"
else:
    print("⚠️ ADVERTENCIA: No se detectó GPU. Se usará CPU (lento).")
    device = "cpu"

C:\Users\Jimena Azocar\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Torchaudio version: 2.9.1+cpu
⚠️ ADVERTENCIA: No se detectó GPU. Se usará CPU (lento).


In [3]:
# Desinstalar la versión CPU actual
!pip uninstall -y torch torchaudio

Found existing installation: torch 2.9.1
Uninstalling torch-2.9.1:
  Successfully uninstalled torch-2.9.1
Found existing installation: torchaudio 2.9.1
Uninstalling torchaudio-2.9.1:
  Successfully uninstalled torchaudio-2.9.1


You can safely remove it manually.


In [4]:
# Instalar versión específica para GPU (CUDA 12.1)
!pip install torch torchaudio --index-url https://download.pytorch.org/whl/cu121

Looking in indexes: https://download.pytorch.org/whl/cu121
     ---------------------------------------- 0.0/2.4 GB ? eta -:--:--
     ---------------------------------------- 0.0/2.4 GB 1.7 MB/s eta 0:24:43
     ---------------------------------------- 0.0/2.4 GB 2.2 MB/s eta 0:18:18
     ---------------------------------------- 0.0/2.4 GB 2.9 MB/s eta 0:14:15
     ---------------------------------------- 0.0/2.4 GB 3.6 MB/s eta 0:11:15
     ---------------------------------------- 0.0/2.4 GB 3.7 MB/s eta 0:11:04
     ---------------------------------------- 0.0/2.4 GB 3.7 MB/s eta 0:11:04
     ---------------------------------------- 0.0/2.4 GB 2.4 MB/s eta 0:17:03
     ---------------------------------------- 0.0/2.4 GB 2.4 MB/s eta 0:17:03
     ---------------------------------------- 0.0/2.4 GB 2.5 MB/s eta 0:16:04
     ---------------------------------------- 0.0/2.4 GB 2.4 MB/s eta 0:17:00
     ---------------------------------------- 0.0/2.4 GB 2.4 MB/s eta 0:17:00
     -------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: C:\Users\Jimena Azocar\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip



     --------------------------- ------------ 1.7/2.4 GB 2.3 MB/s eta 0:05:43
     --------------------------- ------------ 1.7/2.4 GB 2.2 MB/s eta 0:05:47
     --------------------------- ------------ 1.7/2.4 GB 2.2 MB/s eta 0:05:45
     --------------------------- ------------ 1.7/2.4 GB 2.2 MB/s eta 0:05:49
     --------------------------- ------------ 1.7/2.4 GB 2.2 MB/s eta 0:05:45
     --------------------------- ------------ 1.7/2.4 GB 2.3 MB/s eta 0:05:41
     --------------------------- ------------ 1.7/2.4 GB 2.2 MB/s eta 0:05:44
     --------------------------- ------------ 1.7/2.4 GB 2.3 MB/s eta 0:05:37
     --------------------------- ------------ 1.7/2.4 GB 2.3 MB/s eta 0:05:39
     --------------------------- ------------ 1.7/2.4 GB 2.3 MB/s eta 0:05:42
     --------------------------- ------------ 1.7/2.4 GB 2.3 MB/s eta 0:05:42
     --------------------------- ------------ 1.7/2.4 GB 2.3 MB/s eta 0:05:42
     --------------------------- ------------ 1.7/2.4 GB 2.3 MB

In [1]:
import torch
import os

# Verificar GPU
print(f"Versión de Torch: {torch.__version__}")
if torch.cuda.is_available():
    print(f"🔥 ÉXITO: GPU Detectada -> {torch.cuda.get_device_name(0)}")
    print("   ¡Ahora tu entrenamiento volará! 🚀")
else:
    print("⚠️ Sigue sin detectar la GPU. Avísame y probamos otra versión.")

Versión de Torch: 2.5.1+cu121
🔥 ÉXITO: GPU Detectada -> NVIDIA GeForce RTX 3050 Ti Laptop GPU
   ¡Ahora tu entrenamiento volará! 🚀


In [2]:
# --- RUTAS ---
model_id = "facebook/mms-300m" 
dataset_path = "./mms_finetuning_data"
processor_path = "./mms-finetune-guahibo"
output_dir = "./modelo_guahibo_gpu" # Le cambié el nombre para diferenciarlo

print(f"📂 Configuración lista.\n   Datos: {dataset_path}\n   Salida: {output_dir}")

📂 Configuración lista.
   Datos: ./mms_finetuning_data
   Salida: ./modelo_guahibo_gpu


In [3]:
# Cargar procesador y datos
try:
    processor = Wav2Vec2Processor.from_pretrained(processor_path)
    dataset = load_from_disk(dataset_path)
    print("✅ Datos y Vocabulario cargados correctamente.")
except Exception as e:
    print(f"❌ Error: {e}")

❌ Error: name 'Wav2Vec2Processor' is not defined


In [4]:
# --- BLOQUE DE RECUPERACIÓN ---
# Ejecuta esto para restaurar las librerías y rutas tras el reinicio

import os
import torch
import numpy as np
import evaluate
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Union
from datasets import load_from_disk
from transformers import (
    Wav2Vec2ForCTC,
    Wav2Vec2Processor,
    Trainer,
    TrainingArguments,
    set_seed
)

# 1. Configuración Básica
set_seed(42)

# 2. Rutas (Asegúrate que sean las mismas)
model_id = "facebook/mms-300m" 
dataset_path = "./mms_finetuning_data"
processor_path = "./mms-finetune-guahibo"
output_dir = "./modelo_guahibo_gpu"

# 3. Verificación Final de GPU
print(f"Torchaudio version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"🔥 ESTADO: GPU LISTA -> {torch.cuda.get_device_name(0)}")
    device = "cuda"
else:
    print("⚠️ ESTADO: Usando CPU")
    device = "cpu"

print("\n✅ Entorno restaurado. Ahora puedes ejecutar la Celda 4.")

C:\Users\Jimena Azocar\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Torchaudio version: 2.5.1+cu121
🔥 ESTADO: GPU LISTA -> NVIDIA GeForce RTX 3050 Ti Laptop GPU

✅ Entorno restaurado. Ahora puedes ejecutar la Celda 4.


In [5]:
# Cargar procesador y datos
try:
    processor = Wav2Vec2Processor.from_pretrained(processor_path)
    dataset = load_from_disk(dataset_path)
    print("✅ Datos y Vocabulario cargados correctamente.")
except Exception as e:
    print(f"❌ Error: {e}")

❌ Error: Can't load feature extractor for './mms-finetune-guahibo'. If you were trying to load it from 'https://huggingface.co/models', make sure you don't have a local directory with the same name. Otherwise, make sure './mms-finetune-guahibo' is the correct path to a directory containing a preprocessor_config.json file


C:\Users\Jimena Azocar\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\transformers\models\wav2vec2\processing_wav2vec2.py:61: FutureWarning: Loading a tokenizer inside Wav2Vec2Processor from a config that does not include a `tokenizer_class` attribute is deprecated and will be removed in v5. Please add `'tokenizer_class': 'Wav2Vec2CTCTokenizer'` attribute to either your `config.json` or `tokenizer_config.json` file to suppress this warning: 
  warnings.warn(


In [6]:
# --- CELDA DE REPARACIÓN DE ARCHIVOS ---
from transformers import Wav2Vec2CTCTokenizer, Wav2Vec2FeatureExtractor, Wav2Vec2Processor
import os
import json

# Rutas
vocab_file = "./vocab.json"
processor_path = "./mms-finetune-guahibo"

# 1. Verificar si existe el vocabulario
if not os.path.exists(vocab_file):
    print("❌ ERROR CRÍTICO: No encuentro el archivo 'vocab.json'.")
    print("   Por favor, asegúrate de haber ejecutado 'auditar_vocabulario.py' anteriormente.")
else:
    print("✅ Vocabulario encontrado.")

    # 2. Reconstruir el Tokenizador
    tokenizer = Wav2Vec2CTCTokenizer(
        vocab_file,
        unk_token="[UNK]",
        pad_token="[PAD]",
        word_delimiter_token="|"
    )

    # 3. Reconstruir el Extractor de Características (Audio)
    feature_extractor = Wav2Vec2FeatureExtractor(
        feature_size=1,
        sampling_rate=16000,
        padding_value=0.0,
        do_normalize=True,
        return_attention_mask=True
    )

    # 4. Crear el Procesador completo
    processor = Wav2Vec2Processor(feature_extractor=feature_extractor, tokenizer=tokenizer)

    # 5. Guardar los archivos faltantes
    if not os.path.exists(processor_path):
        os.makedirs(processor_path)
    
    processor.save_pretrained(processor_path)
    
    print(f"✅ ¡REPARADO! Archivos de configuración guardados en: {processor_path}")
    print("👉 Ahora vuelve a ejecutar la 'Celda 4'. Ya no debería fallar.")

✅ Vocabulario encontrado.
✅ ¡REPARADO! Archivos de configuración guardados en: ./mms-finetune-guahibo
👉 Ahora vuelve a ejecutar la 'Celda 4'. Ya no debería fallar.


In [7]:
# Cargar procesador y datos
try:
    processor = Wav2Vec2Processor.from_pretrained(processor_path)
    dataset = load_from_disk(dataset_path)
    print("✅ Datos y Vocabulario cargados correctamente.")
except Exception as e:
    print(f"❌ Error: {e}")

✅ Datos y Vocabulario cargados correctamente.


In [8]:
def prepare_dataset(batch):
    audio = batch["audio"]
    batch["input_values"] = processor(audio["array"], sampling_rate=audio["sampling_rate"]).input_values[0]
    with processor.as_target_processor():
        batch["input_ids"] = processor(batch["sentence"]).input_ids
    return batch

print("⚙️ Procesando dataset para GPU...")
dataset_processed = dataset.map(prepare_dataset, remove_columns=dataset["train"].column_names, num_proc=1)
print("✅ ¡Listo!")

⚙️ Procesando dataset para GPU...


Map (num_proc=1):   0%|                                                                                    | 0/945 [00:08<?, ? examples/s]


ImportError: To support decoding audio data, please install 'torchcodec'.

In [9]:
# Instalar las librerías de decodificación de audio que faltan
!pip install torchcodec soundfile librosa

  Using cached soundfile-0.13.1-py2.py3-none-win_amd64.whl.metadata (16 kB)
  Using cached librosa-0.11.0-py3-none-any.whl.metadata (8.7 kB)
  Using cached pooch-1.8.2-py3-none-any.whl.metadata (10 kB)
   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
    --------------------------------------- 0.0/2.2 MB 262.6 kB/s eta 0:00:09
    --------------------------------------- 0.0/2.2 MB 279.3 kB/s eta 0:00:08
   -- ------------------------------------- 0.1/2.2 MB 595.3 kB/s eta 0:00:04
   -- ------------------------------------- 0.1/2.2 MB 655.8 kB/s eta 0:00:04
   ---- ----------------------------------- 0.2/2.2 MB 846.9 kB/s eta 0:00:03
   ------ --------------------------------- 0.4/2.2 MB 1.1 MB/s eta 0:00:02
   ------ --------------------------------- 0.4/2.2 MB 1.1 MB/s eta 0:00:02
   ------- -------------------------------- 0.4/2


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: C:\Users\Jimena Azocar\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [1]:
import os
import torch
import numpy as np
import evaluate
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Union
from datasets import load_from_disk
from transformers import (
    Wav2Vec2ForCTC,
    Wav2Vec2Processor,
    Trainer,
    TrainingArguments,
    set_seed
)

set_seed(42)
model_id = "facebook/mms-300m" 
dataset_path = "./mms_finetuning_data"
processor_path = "./mms-finetune-guahibo"
output_dir = "./modelo_guahibo_gpu"

if torch.cuda.is_available():
    print(f"🔥 GPU LISTA: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ CPU")

# Cargar procesador y datos de nuevo
processor = Wav2Vec2Processor.from_pretrained(processor_path)
dataset = load_from_disk(dataset_path)
print("✅ Entorno y Datos recargados.")

C:\Users\Jimena Azocar\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🔥 GPU LISTA: NVIDIA GeForce RTX 3050 Ti Laptop GPU
✅ Entorno y Datos recargados.


In [2]:
def prepare_dataset(batch):
    audio = batch["audio"]
    batch["input_values"] = processor(audio["array"], sampling_rate=audio["sampling_rate"]).input_values[0]
    with processor.as_target_processor():
        batch["input_ids"] = processor(batch["sentence"]).input_ids
    return batch

print("⚙️ Procesando dataset para GPU...")
dataset_processed = dataset.map(prepare_dataset, remove_columns=dataset["train"].column_names, num_proc=1)
print("✅ ¡Listo!")

⚙️ Procesando dataset para GPU...


Map (num_proc=1):   0%|                                                                                    | 0/945 [00:07<?, ? examples/s]


RuntimeError: Could not load libtorchcodec. Likely causes:
          1. FFmpeg is not properly installed in your environment. We support
             versions 4, 5, 6, 7, and 8. On Windows, ensure you've installed
             the "full-shared" version which ships DLLs.
          2. The PyTorch version (2.5.1+cu121) is not compatible with
             this version of TorchCodec. Refer to the version compatibility
             table:
             https://github.com/pytorch/torchcodec?tab=readme-ov-file#installing-torchcodec.
          3. Another runtime dependency; see exceptions below.
        The following exceptions were raised as we tried to load libtorchcodec:
        
[start of libtorchcodec loading traceback]
FFmpeg version 8: Could not find module 'C:\Users\Jimena Azocar\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torchcodec\libtorchcodec_core8.dll' (or one of its dependencies). Try using the full pa

In [3]:
# 1. Desinstalar la librería que pide FFmpeg
!pip uninstall -y torchcodec

# 2. Reinstalar la librería clásica que funciona siempre
!pip install soundfile librosa

Found existing installation: torchcodec 0.9.1
Uninstalling torchcodec-0.9.1:
  Successfully uninstalled torchcodec-0.9.1



[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: C:\Users\Jimena Azocar\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [1]:
import os
import torch
import numpy as np
import evaluate
from dataclasses import dataclass
from typing import Any, Dict, List, Union
from datasets import load_from_disk, Audio
from transformers import (
    Wav2Vec2ForCTC,
    Wav2Vec2Processor,
    Trainer,
    TrainingArguments,
    set_seed
)

# 1. Configuración
set_seed(42)
model_id = "facebook/mms-300m" 
dataset_path = "./mms_finetuning_data"
processor_path = "./mms-finetune-guahibo"
output_dir = "./modelo_guahibo_gpu"

print(f"🔥 GPU: {torch.cuda.get_device_name(0)}" if torch.cuda.is_available() else "⚠️ CPU")

# 2. Cargar Procesador y Datos
print("📂 Cargando recursos...")
processor = Wav2Vec2Processor.from_pretrained(processor_path)
dataset = load_from_disk(dataset_path)

# 3. Forzar el uso de SoundFile (el truco para que no pida torchcodec)
# Convertimos la columna de audio para asegurar compatibilidad
dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))

# 4. Función de Preprocesamiento
def prepare_dataset(batch):
    audio = batch["audio"]
    batch["input_values"] = processor(audio["array"], sampling_rate=16000).input_values[0]
    with processor.as_target_processor():
        batch["input_ids"] = processor(batch["sentence"]).input_ids
    return batch

# 5. Ejecutar Mapeo
print("⚙️ Procesando audios para GPU (ahora usando SoundFile)...")
# Usamos num_proc=1 para evitar errores de procesos en Windows
dataset_processed = dataset.map(prepare_dataset, remove_columns=dataset["train"].column_names, num_proc=1)

print("✅ ¡ÉXITO! Datos procesados correctamente.")
print(f"   Muestras de entrenamiento: {len(dataset_processed['train'])}")

C:\Users\Jimena Azocar\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🔥 GPU: NVIDIA GeForce RTX 3050 Ti Laptop GPU
📂 Cargando recursos...
⚙️ Procesando audios para GPU (ahora usando SoundFile)...


Map (num_proc=1):   0%|                                                                                    | 0/945 [00:07<?, ? examples/s]


ImportError: To support decoding audio data, please install 'torchcodec'.

In [2]:
# 1. Borrar a la fuerza las librerías conflictivas
!pip uninstall -y torchcodec
!pip uninstall -y datasets

# 2. Reinstalar datasets limpio y soundfile
!pip install datasets soundfile librosa

Found existing installation: datasets 4.4.2
Uninstalling datasets-4.4.2:
  Successfully uninstalled datasets-4.4.2
  Using cached datasets-4.4.2-py3-none-any.whl.metadata (19 kB)
Using cached datasets-4.4.2-py3-none-any.whl (512 kB)


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: C:\Users\Jimena Azocar\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [1]:
import os
import torch
import numpy as np
import evaluate
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Union
from datasets import load_from_disk, Audio
from transformers import (
    Wav2Vec2ForCTC,
    Wav2Vec2Processor,
    Trainer,
    TrainingArguments,
    set_seed
)

# 1. Configuración y Semilla
set_seed(42)
model_id = "facebook/mms-300m" 
dataset_path = "./mms_finetuning_data"
processor_path = "./mms-finetune-guahibo"
output_dir = "./modelo_guahibo_gpu"

# 2. Verificar GPU
print(f"🔥 GPU: {torch.cuda.get_device_name(0)}" if torch.cuda.is_available() else "⚠️ CPU")

# 3. Cargar Procesador y Datos
print("📂 Cargando recursos...")
try:
    processor = Wav2Vec2Processor.from_pretrained(processor_path)
    dataset = load_from_disk(dataset_path)
    
    # --- EL TRUCO CLAVE ---
    # Forzamos explícitamente a que el audio se decodifique, evitando torchcodec
    print("🔧 Ajustando backend de audio...")
    dataset = dataset.cast_column("audio", Audio(sampling_rate=16000, decode=True))
    
    print("✅ Datos cargados.")
except Exception as e:
    print(f"❌ Error cargando: {e}")

# 4. Función de Preprocesamiento
def prepare_dataset(batch):
    audio = batch["audio"]
    # Al acceder a batch["audio"], 'datasets' intentará decodificar.
    # Si torchcodec está bien borrado, usará soundfile.
    batch["input_values"] = processor(audio["array"], sampling_rate=16000).input_values[0]
    
    with processor.as_target_processor():
        batch["input_ids"] = processor(batch["sentence"]).input_ids
    return batch

# 5. Ejecutar Mapeo
print("⚙️ Procesando audios para GPU...")
# IMPORTANTE: num_proc=1 es vital en Windows para evitar errores de proceso
dataset_processed = dataset.map(prepare_dataset, remove_columns=dataset["train"].column_names, num_proc=1)

print("✅ ¡ÉXITO TOTAL! Datos procesados.")
print(f"   Muestras listas para entrenar: {len(dataset_processed['train'])}")

C:\Users\Jimena Azocar\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🔥 GPU: NVIDIA GeForce RTX 3050 Ti Laptop GPU
📂 Cargando recursos...
🔧 Ajustando backend de audio...
✅ Datos cargados.
⚙️ Procesando audios para GPU...


Map (num_proc=1):   0%|                                                                                    | 0/945 [00:13<?, ? examples/s]


ImportError: To support decoding audio data, please install 'torchcodec'.

In [2]:
# 1. Desinstalar la versión problemática
!pip uninstall -y datasets

# 2. Instalar la versión estable (2.19.0) que funciona perfecto en Windows
!pip install datasets==2.19.0 soundfile librosa

Found existing installation: datasets 4.4.2
Uninstalling datasets-4.4.2:
  Successfully uninstalled datasets-4.4.2
INFO: pip is looking at multiple versions of multiprocess to determine which version is compatible with other requirements. This could take a while.
   ---------------------------------------- 0.0/542.0 kB ? eta -:--:--
   -- ------------------------------------- 30.7/542.0 kB 1.3 MB/s eta 0:00:01
   -- ------------------------------------ 41.0/542.0 kB 495.5 kB/s eta 0:00:02
   ------- ------------------------------ 112.6/542.0 kB 939.4 kB/s eta 0:00:01
   -------------- ------------------------- 194.6/542.0 kB 1.2 MB/s eta 0:00:01
   --------------------- ------------------ 286.7/542.0 kB 1.5 MB/s eta 0:00:01
   ----------------------- ---------------- 317.4/542.0 kB 1.2 MB/s eta 0:00:01
   ----------------------------------- ---- 481.3/542.0 kB 1.6 MB/s eta 0:00:01
   ---------------------------------------  532.5/542.0 kB 1.6 MB/s eta 0:00:01
   -----------------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: C:\Users\Jimena Azocar\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [1]:
import os
import torch
import numpy as np
import evaluate
from dataclasses import dataclass
from typing import Any, Dict, List, Union
from datasets import load_from_disk, Audio
from transformers import (
    Wav2Vec2ForCTC,
    Wav2Vec2Processor,
    Trainer,
    TrainingArguments,
    set_seed
)

# 1. Configuración
set_seed(42)
model_id = "facebook/mms-300m" 
dataset_path = "./mms_finetuning_data"
processor_path = "./mms-finetune-guahibo"
output_dir = "./modelo_guahibo_gpu"

print(f"🔥 GPU: {torch.cuda.get_device_name(0)}" if torch.cuda.is_available() else "⚠️ CPU")

# 2. Cargar Procesador y Datos
print("📂 Cargando recursos...")
try:
    processor = Wav2Vec2Processor.from_pretrained(processor_path)
    dataset = load_from_disk(dataset_path)
    
    # Forzamos el formato usando la librería clásica (SoundFile)
    print("🔧 Ajustando audio...")
    dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))
    
    print(f"✅ Datos cargados: {len(dataset['train'])} muestras.")
except Exception as e:
    print(f"❌ Error cargando: {e}")

# 3. Función de Preprocesamiento
def prepare_dataset(batch):
    audio = batch["audio"]
    # Al ser la versión 2.19.0, esto usará soundfile automáticamente sin pedir torchcodec
    batch["input_values"] = processor(audio["array"], sampling_rate=16000).input_values[0]
    
    with processor.as_target_processor():
        batch["input_ids"] = processor(batch["sentence"]).input_ids
    return batch

# 4. Ejecutar Mapeo
print("⚙️ Procesando audios para GPU...")
# num_proc=1 es clave en Windows
dataset_processed = dataset.map(prepare_dataset, remove_columns=dataset["train"].column_names, num_proc=1)

print("✅ ¡ÉXITO TOTAL! Datos procesados y listos para entrenar.")

C:\Users\Jimena Azocar\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🔥 GPU: NVIDIA GeForce RTX 3050 Ti Laptop GPU
📂 Cargando recursos...
🔧 Ajustando audio...
✅ Datos cargados: 945 muestras.
⚙️ Procesando audios para GPU...


Map:   0%|                                                                                                 | 0/945 [00:00<?, ? examples/s]C:\Users\Jimena Azocar\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\transformers\models\wav2vec2\processing_wav2vec2.py:180: UserWarning: `as_target_processor` is deprecated and will be removed in v5 of Transformers. You can process your labels by using the argument `text` of the regular `__call__` method (either in the same call as your audio inputs, or in a separate call.
  warnings.warn(
Map: 100%|████████████████████████████████████████████████████████████████████████████████████████| 35/35 [00:00<00:00, 289.26 examples/s]

✅ ¡ÉXITO TOTAL! Datos procesados y listos para entrenar.


In [2]:
@dataclass
class DataCollatorCTCWithPadding:
    processor: Wav2Vec2Processor
    padding: Union[bool, str] = True

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_values": feature["input_values"]} for feature in features]
        label_features = [{"input_ids": feature["input_ids"]} for feature in features]

        batch = self.processor.feature_extractor.pad(input_features, padding=self.padding, return_tensors="pt")
        with self.processor.as_target_processor():
            labels_batch = self.processor.tokenizer.pad(label_features, padding=self.padding, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        batch["labels"] = labels
        return batch

data_collator = DataCollatorCTCWithPadding(processor=processor, padding=True)
print("✅ Data Collator configurado.")

✅ Data Collator configurado.


In [4]:
wer_metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_logits = pred.predictions
    pred_ids = np.argmax(pred_logits, axis=-1)
    pred.label_ids[pred.label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str = processor.batch_decode(pred_ids)
    label_str = processor.batch_decode(pred.label_ids, group_tokens=False)
    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}

In [5]:
print(f"🧠 Cargando modelo {model_id} en la GPU...")

model = Wav2Vec2ForCTC.from_pretrained(
    model_id,
    ctc_loss_reduction="mean",
    pad_token_id=processor.tokenizer.pad_token_id,
    vocab_size=len(processor.tokenizer),
    ignore_mismatched_sizes=True
)

model.freeze_feature_extractor()
model.to("cuda") # Enviamos el modelo a la tarjeta gráfica explícitamente

# --- HIPERPARÁMETROS PARA GPU ---
training_args = TrainingArguments(
    output_dir=output_dir,
    group_by_length=True,
    per_device_train_batch_size=4,  # Tu 3050 Ti tiene 4GB VRAM. '4' es un buen número. Si da error, baja a 2.
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    evaluation_strategy="steps",
    num_train_epochs=15,            # Con GPU podemos permitirnos 15 o 30 épocas rápido.
    fp16=True,                      # 🔥 EL TURBO: Acelera x2 el entrenamiento en RTX
    save_steps=200,
    eval_steps=200,
    logging_steps=50,
    learning_rate=1e-4,
    warmup_steps=100,
    save_total_limit=2,
)

trainer = Trainer(
    model=model,
    data_collator=data_collator,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=dataset_processed["train"],
    eval_dataset=dataset_processed["test_synthetic"],
    tokenizer=processor.feature_extractor,
)

print("✅ Modelo cargado y configurado en NVIDIA RTX.")

🧠 Cargando modelo facebook/mms-300m en la GPU...


C:\Users\Jimena Azocar\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Jimena Azocar\.cache\huggingface\hub\models--facebook--mms-300m. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


ValueError: Due to a serious vulnerability issue in `torch.load`, even with `weights_only=True`, we now require users to upgrade torch to at least v2.6 in order to use the function. This version restriction does not apply when loading files with safetensors.
See the vulnerability report here https://nvd.nist.gov/vuln/detail/CVE-2025-32434

In [1]:
# Instalar una versión de transformers que no bloquee tu PyTorch
!pip install transformers==4.46.0 accelerate==1.0.1

     ---------------------------------------- 0.0/44.1 kB ? eta -:--:--
     --------- ------------------------------ 10.2/44.1 kB ? eta -:--:--
     --------- ------------------------------ 10.2/44.1 kB ? eta -:--:--
     ----------------- -------------------- 20.5/44.1 kB 108.9 kB/s eta 0:00:01
     ----------------- -------------------- 20.5/44.1 kB 108.9 kB/s eta 0:00:01
     ----------------------------------- -- 41.0/44.1 kB 163.4 kB/s eta 0:00:01
     -------------------------------------- 44.1/44.1 kB 154.8 kB/s eta 0:00:00
   ---------------------------------------- 0.0/10.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/10.0 MB 960.0 kB/s eta 0:00:11
   ---------------------------------------- 0.1/10.0 MB 656.4 kB/s eta 0:00:16
   ---------------------------------------- 0.1/10.0 MB 930.9 kB/s eta 0:00:11
    --------------------------------------- 0.2/10.0 MB 1.3 MB/s eta 0:00:08
   - -------------------------------------- 0.3/10.0 MB 1.1 MB/s eta 0:00:09
 

Reason for being yanked: This version unfortunately does not work with 3.8 but we did not drop the support yet
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: C:\Users\Jimena Azocar\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [1]:
import os
import torch
import numpy as np
import evaluate
from dataclasses import dataclass
from datasets import load_from_disk, Audio
from transformers import (
    Wav2Vec2ForCTC,
    Wav2Vec2Processor,
    Trainer,
    TrainingArguments,
    set_seed
)

# --- 1. CONFIGURACIÓN ---
set_seed(42)
model_id = "facebook/mms-300m" 
dataset_path = "./mms_finetuning_data"
processor_path = "./mms-finetune-guahibo"
output_dir = "./modelo_guahibo_gpu"

print(f"🔥 GPU: {torch.cuda.get_device_name(0)}" if torch.cuda.is_available() else "⚠️ CPU (Algo anda mal)")

# --- 2. CARGAR DATOS ---
print("📂 Cargando procesador y datos...")
processor = Wav2Vec2Processor.from_pretrained(processor_path)
dataset = load_from_disk(dataset_path)

# Ajuste de audio (SoundFile)
dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))

# Preprocesamiento rápido
def prepare_dataset(batch):
    audio = batch["audio"]
    batch["input_values"] = processor(audio["array"], sampling_rate=16000).input_values[0]
    with processor.as_target_processor():
        batch["input_ids"] = processor(batch["sentence"]).input_ids
    return batch

print("⚙️ Procesando datos...")
dataset_processed = dataset.map(prepare_dataset, remove_columns=dataset["train"].column_names, num_proc=1)

# --- 3. CONFIGURAR ENTRENAMIENTO ---
@dataclass
class DataCollatorCTCWithPadding:
    processor: Wav2Vec2Processor
    padding: bool = True

    def __call__(self, features):
        input_features = [{"input_values": feature["input_values"]} for feature in features]
        label_features = [{"input_ids": feature["input_ids"]} for feature in features]
        
        batch = self.processor.feature_extractor.pad(input_features, padding=self.padding, return_tensors="pt")
        with self.processor.as_target_processor():
            labels_batch = self.processor.tokenizer.pad(label_features, padding=self.padding, return_tensors="pt")
            
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        batch["labels"] = labels
        return batch

wer_metric = evaluate.load("wer")
def compute_metrics(pred):
    pred_logits = pred.predictions
    pred_ids = np.argmax(pred_logits, axis=-1)
    pred.label_ids[pred.label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str = processor.batch_decode(pred_ids)
    label_str = processor.batch_decode(pred.label_ids, group_tokens=False)
    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}

print("🧠 Cargando modelo en GPU...")
model = Wav2Vec2ForCTC.from_pretrained(
    model_id,
    ctc_loss_reduction="mean",
    pad_token_id=processor.tokenizer.pad_token_id,
    vocab_size=len(processor.tokenizer),
    ignore_mismatched_sizes=True
)
model.freeze_feature_extractor()
model.to("cuda")

training_args = TrainingArguments(
    output_dir=output_dir,
    group_by_length=True,
    per_device_train_batch_size=2, # Optimizado para RTX 3050 Ti
    gradient_accumulation_steps=4,
    evaluation_strategy="steps",
    num_train_epochs=15,
    fp16=True,                     # ACELERACIÓN ACTIVADA
    save_steps=200,
    eval_steps=200,
    logging_steps=50,
    learning_rate=1e-4,
    warmup_steps=100,
    save_total_limit=2,
    report_to="none"
)

trainer = Trainer(
    model=model,
    data_collator=DataCollatorCTCWithPadding(processor=processor, padding=True),
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=dataset_processed["train"],
    eval_dataset=dataset_processed["train"].select(range(10)),
    tokenizer=processor.feature_extractor,
)

# --- 4. ¡PLAY! ---
print("\n🚀 --- INICIANDO EL ENTRENAMIENTO --- 🚀")
trainer.train()

# Guardar al final
print("💾 Guardando modelo final...")
trainer.save_model(output_dir)
processor.save_pretrained(output_dir)
print("✨ ¡TERMINADO! ✨")

C:\Users\Jimena Azocar\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.
0it [00:00, ?it/s]


🔥 GPU: NVIDIA GeForce RTX 3050 Ti Laptop GPU
📂 Cargando procesador y datos...
⚙️ Procesando datos...


Map:   0%|                                                                                                 | 0/945 [00:00<?, ? examples/s]C:\Users\Jimena Azocar\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\transformers\models\wav2vec2\processing_wav2vec2.py:174: UserWarning: `as_target_processor` is deprecated and will be removed in v5 of Transformers. You can process your labels by using the argument `text` of the regular `__call__` method (either in the same call as your audio inputs, or in a separate call.
  warnings.warn(
Map: 100%|████████████████████████████████████████████████████████████████████████████████████████| 35/35 [00:00<00:00, 340.66 examples/s]


🧠 Cargando modelo en GPU...


Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/mms-300m and are newly initialized: ['lm_head.bias', 'lm_head.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\Jimena Azocar\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\transformers\models\wav2vec2\modeling_wav2vec2.py:2178: FutureWarning: The method `freeze_feature_extractor` is deprecated and will be removed in Transformers v5. Please use the equivalent `freeze_feature_encoder` method instead.
  warnings.warn(
C:\Users\Jimena Azocar\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\transformers\training_args.py:1559: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(



🚀 --- INICIANDO EL ENTRENAMIENTO --- 🚀


KeyError: 'input_ids'

In [2]:
# --- REPARACIÓN DEL DATASET ---

# 1. Detectar automáticamente el nombre de la columna de texto
# (A veces se llama 'sentence', 'text' o 'transcription')
columnas = dataset["train"].column_names
columna_texto = "sentence" if "sentence" in columnas else "text"
if "transcription" in columnas: columna_texto = "transcription"

print(f"🔍 Columna de texto detectada: '{columna_texto}'")

# 2. Función de Preprocesamiento (Versión "Limpia")
# Esta versión crea un diccionario nuevo para asegurar que se guarden los datos
def prepare_dataset_robust(batch):
    audio = batch["audio"]
    
    # Procesar Audio
    inputs = processor(audio["array"], sampling_rate=16000)
    
    # Procesar Texto
    with processor.as_target_processor():
        labels = processor(batch[columna_texto]).input_ids
        
    # Retornar SOLO lo que necesitamos
    return {
        "input_values": inputs.input_values[0],
        "input_ids": labels
    }

print("⚙️ Re-procesando datos (Intento robusto)...")

# 3. Aplicar el mapa
dataset_processed = dataset.map(
    prepare_dataset_robust, 
    remove_columns=columnas, # Borramos todo lo viejo
    num_proc=1
)

# 4. VERIFICACIÓN (La prueba de fuego)
print("\n🧐 Verificando columnas resultantes:")
print(dataset_processed["train"].column_names)

if "input_ids" in dataset_processed["train"].column_names:
    print("✅ ¡PROBLEMA RESUELTO! La columna 'input_ids' existe.")
    
    # --- AHORA SÍ, ENTRENAR ---
    # Reconfiguramos el trainer con el dataset corregido
    trainer.train_dataset = dataset_processed["train"]
    trainer.eval_dataset = dataset_processed["train"].select(range(10))
    
    print("\n🚀 --- RE-INICIANDO ENTRENAMIENTO --- 🚀")
    trainer.train()
else:
    print("❌ ERROR: Aún falta la columna 'input_ids'. Avísame.")

🔍 Columna de texto detectada: 'sentence'
⚙️ Re-procesando datos (Intento robusto)...


Map: 100%|████████████████████████████████████████████████████████████████████████████████████████| 35/35 [00:00<00:00, 258.86 examples/s]



🧐 Verificando columnas resultantes:
['input_values', 'input_ids']
✅ ¡PROBLEMA RESUELTO! La columna 'input_ids' existe.

🚀 --- RE-INICIANDO ENTRENAMIENTO --- 🚀


KeyError: 'input_ids'

In [3]:
# --- BLOQUE DE CORRECCIÓN FINAL ---
import torch
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Union

# 1. EL TRUCO MAESTRO: Renombrar la columna 'input_ids' a 'labels'
# Esto evita que el Trainer la borre automáticamente.
print("🔄 Ajustando nombres de columnas...")
if "input_ids" in dataset_processed["train"].column_names:
    dataset_processed = dataset_processed.rename_column("input_ids", "labels")
    print("✅ Columna renombrada a 'labels' correctamente.")
else:
    print("ℹ️ La columna ya se llamaba 'labels' o no existe.")

# 2. Actualizar el Data Collator para que busque 'labels'
@dataclass
class DataCollatorCTCWithPadding:
    processor: Any
    padding: bool = True

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # Entrada de audio
        input_features = [{"input_values": feature["input_values"]} for feature in features]
        
        # Entrada de texto (AHORA BUSCAMOS 'labels')
        label_features = [{"input_ids": feature["labels"]} for feature in features]

        # Padding de audio
        batch = self.processor.feature_extractor.pad(
            input_features,
            padding=self.padding,
            return_tensors="pt",
        )
        
        # Padding de texto
        with self.processor.as_target_processor():
            labels_batch = self.processor.tokenizer.pad(
                label_features,
                padding=self.padding,
                return_tensors="pt",
            )

        # Reemplazar padding con -100 para ignorarlo en el entrenamiento
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        
        batch["labels"] = labels
        return batch

# 3. Re-inicializar el Trainer con el nuevo dataset y collator
trainer.data_collator = DataCollatorCTCWithPadding(processor=processor, padding=True)
trainer.train_dataset = dataset_processed["train"]
trainer.eval_dataset = dataset_processed["train"].select(range(10))

# 4. ¡PLAY! 🔥
print("\n🚀 --- INICIANDO ENTRENAMIENTO (INTENTO FINAL) --- 🚀")
trainer.train()

🔄 Ajustando nombres de columnas...
✅ Columna renombrada a 'labels' correctamente.

🚀 --- INICIANDO ENTRENAMIENTO (INTENTO FINAL) --- 🚀


Step,Training Loss,Validation Loss,Wer
200,13.448500,3.501741,1.000000
400,12.746400,3.323984,1.000000
600,12.121100,3.318400,1.000000
800,11.657700,3.013139,1.000000
1000,10.720700,2.754662,1.000000
1200,9.414700,2.235872,1.000000
1400,8.417800,1.957574,1.000000
1600,7.785400,1.799870,1.000000


Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
C:\Users\Jimena Azocar\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\transformers\models\wav2vec2\processing_wav2vec2.py:174: UserWarning: `as_target_processor` is deprecated and will be removed in v5 of Transformers. You can process your labels by using the argument `text` of the regular `__call__` method (either in the same call as your audio inputs, or in a separate call.
  warnings.warn(
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
C:\Users\Jimena Azocar\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\transformers\models\wav2vec2\process

TrainOutput(global_step=1770, training_loss=12.252126282083113, metrics={'train_runtime': 5667.8611, 'train_samples_per_second': 2.501, 'train_steps_per_second': 0.312, 'total_flos': 5.0027240478745344e+17, 'train_loss': 12.252126282083113, 'epoch': 14.968287526427062})

In [4]:
# Guardar el modelo y el procesador para usarlos después
print("💾 Guardando tu Inteligencia Artificial...")

# Guardar en la carpeta definida
trainer.save_model(output_dir)
processor.save_pretrained(output_dir)

print(f"✨ ¡LISTO! Modelo guardado en: {output_dir}")
print("   Ahora puedes cerrar este notebook y usar el modelo cuando quieras.")

💾 Guardando tu Inteligencia Artificial...
✨ ¡LISTO! Modelo guardado en: ./modelo_guahibo_gpu
   Ahora puedes cerrar este notebook y usar el modelo cuando quieras.


In [5]:
# --- PRUEBA DE INFERENCIA ---
import torch
import random

# 1. Elegir un audio al azar del dataset
sample = dataset_processed["train"][random.randint(0, len(dataset_processed["train"])-1)]

print("🎧 Escuchando audio...")

# 2. Preparar el audio para la GPU
input_values = torch.tensor(sample["input_values"]).unsqueeze(0).to("cuda")

# 3. Predecir (Sin calcular gradientes para ir rápido)
with torch.no_grad():
    logits = model(input_values).logits

# 4. Decodificar (Convertir números a letras)
pred_ids = torch.argmax(logits, dim=-1)
transcription = processor.batch_decode(pred_ids)[0]

# 5. Mostrar resultados
print("-" * 30)
print(f"📝 Lo que dice el modelo (Predicción): \n'{transcription}'")
print("-" * 30)
print(f"✅ Lo que debería decir (Realidad):    \n'{processor.decode(sample['labels'])}'")
print("-" * 30)

🎧 Escuchando audio...
------------------------------
📝 Lo que dice el modelo (Predicción): 
'hótto'
------------------------------
✅ Lo que debería decir (Realidad):    
'robóta'
------------------------------
